# 2-B: One job, three placements: code, text, script

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/en/notebooks/2-B_code_text_script.ipynb)

> 📍 **Matching slides**: Session 2 "Stop 2: Code · Text · Scripts", slides **13-19** (class time 0:28-0:48)

## 🎯 The mini project for these two sessions: the Homework Helper

The **Homework Helper** is a small program that helps check Python homework. You give it a piece of homework, and it replies with the review result.
In the workflow version (1-A, workflow), a syntax error gets an error reply first; only code with valid syntax goes on to the AI, which writes feedback containing **"Strengths:" and "Improvements:"**.
If the format check keeps failing, the flow stops when it reaches the maximum number of generations and marks the result as not passed. "Stopped" must never be read as "passed".

**Keep three things apart:** valid syntax (Python can read it) ≠ runs successfully (no error while running) ≠ meets the assignment (matches what the assignment asks for).

We chose it because it happens to use all three kinds of things (and you can also combine them):

| What the helper needs | Example | What we use |
| --- | --- | --- |
| Steps controlled by a flow | Check syntax first; report errors; only on a pass, write feedback and check its format | A **workflow** defined in code (LangGraph, Session 1) |
| Things that need judgment | How to word the feedback, whether to actually run the code and see | Leave it to the **AI** (agent, Session 1) |
| Ways of working and rules | Should the feedback be gentle or strict, what a good variable name looks like | A **handbook** written for the AI to read (Skill, Session 2). Rules that code can check (such as variable naming) can also come with a script or go into the workflow; 2-B compares them |

👉 **This notebook covers:** Put the same "check variable naming" job in three places (handbook text, a script, and a workflow), run each three times, and compare "who does the check" with "who decides to run it".

<details><summary>Open this when you need to look up a component name (no need to memorize it)</summary>

| Component | What it does, in plain words |
| --- | --- |
| `check_syntax` (check syntax) | Only checks whether the code's "grammar" is right, for example whether brackets are closed or a colon is missing. It **does not run the code** |
| `explain_error` (explain the error) | When the syntax is wrong, tells the student in one sentence which line is wrong |
| `give_feedback` (write feedback) | Once a model is connected in 1-A, asks the AI to write "Strengths + Improvements" feedback; the next station checks whether it matches the format |
| `check_format` (check format) | Checks the order of the two markers, that each part is non-empty, and the total character count. It does not verify that the feedback content is correct |
| Tools: `run_code`, `check_indent`, `read_spec` (from 1-B on); `check_naming` (from 2-B on) | Run the homework under teaching limits, check indentation, read the assignment requirements, check variable names |
| Handbooks: `friendly-feedback`, `strict-review`; `naming-check` (2-B), `detailed-feedback` (the 2-C security experiment) | Tell the AI whether feedback should be gentle and encouraging or strict; the naming-check handbook comes with a script; the last one hides a suspicious instruction so you can practice checking an outside handbook |

</details>

**Reminder:** run "0. Setup" first, then follow each part: setup cell → predict → run → compare.
`%%writefile` means "save this cell's content as a file"; edits only take effect after you run the cell. Re-running a setup cell may overwrite the handbooks, so save your own edits first.

## What this notebook does

The Homework Helper wants to do one more thing: **check that variable names are "lowercase letters plus underscores"** (for example `my_score` passes; `myScore` and `MyScore` do not).
The same rule can live in three places:

| Placement | Where it lives | Who does the check |
| --- | --- | --- |
| **A** | Written into the handbook text (`skills/naming-check/SKILL.version-a.md`) | The AI works out how to check it again every time |
| **B** | Written as a small program next to the handbook, `scripts/check_naming.py`, and the handbook tells the AI to run it | The AI decides whether to run it; the check itself is Python |
| **C** | Written as one station in the flow (LangGraph) | The flow always runs the check when it reaches that station; the AI never touches it |

All three can do the job, so what is the difference? **Run each three times, and record whether the results are the same, whether a tool was called, and how many AI calls were made. When the results are the same, the difference is who checks and who decides to run it.**
This notebook has one new term: **script** (a small program that sits in the handbook's `scripts/` folder and really runs).

| Part | What you do | Matching slides |
| --- | --- | --- |
| Part 1 | Look at the homework and the script; run `check_naming.py` once from the command line | 14, 16 |
| Part 2 | Placement A: written in the handbook text, run three times | 15 |
| Part 3 | Placement B: written as a script, the handbook tells the AI to run it, run three times | 16 |
| Part 4 | Placement C: written as a LangGraph node, run three times | 17 |
| Part 5 | The three versions side by side; this session's rule of thumb | 17-18 |
| Part 6 | Pause: where does each of five jobs go? | 19 |

## How to use it (you do not need to edit anything in this notebook)

1. Run **"0. Environment setup"** first.
2. Every part: **🤔 guess whether the three runs match and how many AI calls → ▶ run → 👀 compare**.
3. Parts 2-4 write this version's latest three-run result to `2B_results.json`; rerunning replaces that version's old record. Part 5 uses the last record of each of A, B, and C, so finish them in order with the same rule and homework.
4. The environment setup and file preparation cells in this notebook overwrite the handbooks and scripts they contain. Keep any edits first, then prepare again; other notebooks do not pick up your edited versions automatically.
5. How many AI calls each run makes: A makes 1 per round; B varies with tool choice and the loop, so read the measured count; C makes 0 for the naming check.

In [ ]:
#@title 0. Setup (run it the first time you open this notebook, or after the runtime is reset)
#@markdown **Model source**: `colab` = the model built into Colab (no API key needed; availability and quota depend on your account); `gemini` = the instructor's Gemini API key (the backup option)
model_source = "colab"  #@param ["colab", "gemini"]
#@markdown If you pick `gemini`: we recommend adding `GOOGLE_API_KEY` under 🔑 "Secrets" in the left sidebar. You can also paste the key below (⚠️ if you paste it, do not share this notebook)
INSTRUCTOR_API_KEY = ""  #@param {type:"string"}
#@markdown Advanced (you usually don't need to change these): set a model name, or leave it empty to use the default
colab_model_name = ""  #@param {type:"string"}
gemini_model_name = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. Install packages with uv (limited to compatible version ranges; time depends on your network)
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ Installing LangGraph / LangChain with uv ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4"', shell=True, check=True)

# 2. Write the shared files this notebook uses
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_files = {
    'llm.py': r'''"""Model adapter: wraps either "Colab's built-in model" or "the instructor's Gemini API key" into one model object.

Students don't need to read this file. Every later cell only writes model.invoke(...) and model.bind_tools(...),
and it works the same no matter which provider is behind it.

- colab : google.colab.ai.generate_text (works with a free account, no API key needed)
- gemini: langchain_google_genai.ChatGoogleGenerativeAI (needs GOOGLE_API_KEY)

generate_text takes only one piece of text and returns only one piece of text. It has no native tool calling.
So ColabAIChat adds a small adapter:
1. It joins the messages into one text (system instructions, user, AI, and tool results each get a header)
2. When tools are bound, it appends the tool list and asks the model to "reply with a single line of JSON if you need a tool"
3. It parses the model's JSON into tool_calls, in exactly the same format as native tool calling
This lets you see that the model only ever "says" which tool it wants.
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

SETTINGS_FILE = Path(__file__).with_name(".llm.json")
DEFAULT_GEMINI_MODEL = "gemini-flash-latest"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: about 4 characters per token for English text.
    This is only a guess. A real tokenizer can count noticeably more or fewer."""
    return max(1, round(len(text) / 4))


# ---------- Join the messages into one text ----------

def _describe_tools(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params}): {desc}")
    return "\n".join(lines)


def _messages_to_text(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"=== System instructions ===\n{m.text}")
        elif m.type == "human":
            parts.append(f"=== User ===\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("=== AI ===\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"=== AI ===\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "tool"
            parts.append(f"=== Result of tool {name} ===\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "This time you must call at least one tool.\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"This time you must call {tool_choice}.\n"
        parts.append(
            "=== Tools you can use ===\n"
            f"{_describe_tools(tools)}\n\n"
            "=== Reply rules ===\n"
            f"{must}"
            "If you need to use a tool, reply with a single line of JSON only. Add no other text and no ``` marks:\n"
            '{"tool_calls": [{"name": "tool name", "args": {"parameter name": "parameter value"}}]}\n'
            "If you don't need a tool (for example, you already have enough information), just answer in plain text and output no JSON."
        )
    parts.append("=== AI ===")
    return "\n\n".join(parts)


def _parse_reply(text: str, tools: list[dict] | None) -> AIMessage:
    """If the model replied with a JSON note, turn it into tool_calls. Otherwise treat it as plain text."""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """Wraps google.colab.ai.generate_text as a LangChain chat model."""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _messages_to_text(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _parse_reply(text, tools)
        message.usage_metadata = {
            "input_tokens": estimate_tokens(prompt),
            "output_tokens": estimate_tokens(text),
            "total_tokens": estimate_tokens(prompt) + estimate_tokens(text),
        }
        message.response_metadata["token_count"] = "estimate (Colab's built-in model does not report usage)"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- Public: settings and getting the model ----------

def load_settings() -> dict:
    try:
        return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_settings(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """Remember which model provider was chosen (the API key is not saved)."""
    SETTINGS_FILE.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or DEFAULT_GEMINI_MODEL},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _find_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = load_settings()
    # In the cloud there is no settings file: use the environment variable WORKSHOP_PROVIDER=gemini (the key is read from GOOGLE_API_KEY)
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _find_api_key()
        if not key:
            raise RuntimeError(
                "You chose gemini but no API key was found. Add GOOGLE_API_KEY under 🔑 Secrets on the left, "
                "or paste the key from the instructor into the form in \"0. Environment setup\", then rerun that cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or DEFAULT_GEMINI_MODEL, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai only works on Google Colab. If you are not on Colab, choose gemini instead.") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""Small helpers shared by the workshop notebooks (students don't need to read this).

- prepare()                     : the first cell of every part calls it to reconnect the model and reload edited files
- sample_homework               : the sample homework in the form dropdown
- record() / print_history()    : save the results that experiment cells name, then list them by title in the last cell
- show_graph()                  : draw a LangGraph as a diagram
- save_to_drive() / download_zip()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, estimate_tokens  # noqa: F401  (so notebooks can write: from workshop import estimate_tokens)

__all__ = [
    "BASE", "prepare", "sample_homework", "load_homework", "record", "print_state", "print_order_slip", "print_history", "clear_history",
    "ModelCallCounter", "show_graph", "save_to_drive", "download_zip", "estimate_tokens",
]

BASE = Path(__file__).resolve().parent
HISTORY_FILE = BASE / "history.jsonl"

sample_homework = {
    "Missing parenthesis": "print('hi'\n",
    "Correct program": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "Empty": "",
    "Variable typo": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "Messy indentation": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "Inconsistent naming": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def load_homework(name: str) -> str:
    """When the form says "My homework", read /content/review-bot/my_homework.py. Otherwise take it from the samples."""
    if name.startswith("My homework"):
        path = BASE / "my_homework.py"
        if not path.exists():
            raise FileNotFoundError('There is no "My homework" yet. Run the cell above that has %%writefile ... my_homework.py first.')
        return path.read_text(encoding="utf-8")
    return sample_homework[name]


def prepare():
    """The start of every part: put review-bot on the import path, reload edited files, and return the model."""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _make_mcp_stdio_work_in_notebook()
    return get_model()


def _make_mcp_stdio_work_in_notebook() -> None:
    """mcp's stdio_client connects the MCP server's error output to sys.stderr.
    On Colab (Linux), sys.stderr has a real file descriptor, so this works. Jupyter on Windows has none, so we redirect to a log file."""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State history ----------

def _serializable(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serializable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def record(title: str, content) -> None:
    """Save this result (usually a State) into history.jsonl."""
    row = {"time": _dt.datetime.now().strftime("%H:%M:%S"), "title": title, "content": _serializable(content)}
    with HISTORY_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _short(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"... ({len(text) - n} more characters)"


def print_state(state: dict, indent: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{indent}{k}: ({len(v)} messages)")
            for m in v:
                m = _serializable(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{indent}  [{m['type']}] calls {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{indent}  [{m.get('type')}] {_short(m.get('content', ''), 200)}")
        else:
            print(f"{indent}{k}: {_short(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def print_order_slip(step_no: int, state: dict, prev_state: dict | None = None, skip_keys: tuple = ("code",)) -> None:
    """stream_mode="values" gives the whole order slip at every step. This also marks which fields this step newly wrote."""
    if not prev_state:
        note = "start: blank order slip"
    else:
        changed_keys = [k for k in state if state.get(k) != prev_state.get(k)]
        note = "newly written: " + (", ".join(changed_keys) if changed_keys else "(nothing changed)")
    print(f"Step {step_no} ({note})")
    print("  ", {k: (_short(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in skip_keys})


def print_history() -> None:
    if not HISTORY_FILE.exists():
        print("There are no records yet. Run a few parts above first, then come back to this cell.")
        return
    rows = [json.loads(line) for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 {len(rows)} records (saved in {HISTORY_FILE})\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['time']}] {row['title']} =====")
        content = row["content"]
        if isinstance(content, dict):
            print_state(content)
        elif isinstance(content, list):
            for j, item in enumerate(content, 1):
                print(f"  --- step {j} ---")
                if isinstance(item, dict):
                    print_state(item, "    ")
                else:
                    print(f"    {_short(item)}")
        else:
            print(f"  {_short(content, 1000)}")
        print()


def clear_history() -> None:
    HISTORY_FILE.unlink(missing_ok=True)


# ---------- Count how many times the model was called ----------

class ModelCallCounter(BaseCallbackHandler):
    """Put it in config={"callbacks": [counter]}. It adds 1 every time the model is called."""

    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.count += 1


# ---------- Draw the graph ----------

def show_graph(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("(The drawing service is unreachable for now, so here is a text version. You can paste it into https://mermaid.live to see the diagram.)\n")
        print(g.draw_mermaid())


# ---------- Saving ----------

def save_to_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, destination, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ Saved to Google Drive: {destination}")
    print('   (Save the notebook itself with the menu above: "File → Save a copy in Drive")')


def download_zip() -> None:
    from google.colab import files

    name = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(name), "zip", BASE.parent, BASE.name)
    print(f"✅ Packed: {zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""The Homework Helper's toolbox: run_code, check_indent, read_spec, check_naming."""
''',
    'tools/check_naming.py': r'''"""check_naming: wraps the Skill's script scripts/check_naming.py as an agent tool.

Wrap the script in @tool and it joins the agent's toolbox.
In this example the AI decides whether to call the tool and tidies up the result. With the same version and the same input, the Python script returns the same check result,
but you still need to confirm whether the model actually called it and whether it kept the result intact.
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

SCRIPT = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def run_naming_script(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(SCRIPT)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    output = proc.stdout.decode("utf-8").strip()
    return output or "The script failed to run: " + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """Check whether variable and function names are lowercase with underscores (snake_case), and return the lines that break the rule.

    When to use it: when reviewing a homework and you need to list the naming problems (this is the script that comes with the naming-check handbook).
    Parameter code: the complete Python code (a string).
    Returns: one line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)". If everything passes it returns "OK: ...".
    """
    return run_naming_script(code)
''',
    'skills/naming-check/scripts/check_naming.py': r'''"""Check whether variable and function names are lowercase English with underscores (snake_case).

Usage:
    python check_naming.py homework.py        # check one file
    python check_naming.py < homework.py      # or read from standard input
Output:
    One line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)"
    When everything passes it prints "OK: all variable and function names are lowercase with underscores"

A script does one thing and prints only a few short lines, so people and the AI can both check it. The same input and the same script give the same result.
"""

import ast
import re
import sys

RULE = re.compile(r"^_?[a-z][a-z0-9_]*$")


def suggest_name(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def find_all_names(tree: ast.AST) -> dict[str, int]:
    """name → line number of its first appearance (variables, functions, parameters; class names are not included)."""
    names = {}

    def note(name, line):
        if name not in names or line < names[name]:
            names[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            note(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            note(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            note(node.arg, node.lineno)
    return names


def check_names(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"Cannot check: line {e.lineno} has a syntax error"
    problems = [
        f"Line {line}: {name} breaks the rule (suggest {suggest_name(name)})"
        for name, line in sorted(find_all_names(tree).items(), key=lambda x: (x[1], x[0]))
        if not RULE.match(name)
    ]
    return "\n".join(problems) if problems else "OK: all variable and function names are lowercase with underscores"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(check_names(source))
''',
    'skills/naming-check/SKILL.md': r'''---
name: naming-check
description: Checks whether the variable and function names in a Python homework are lowercase English with underscores (snake_case). Use it when reviewing a homework and you need to list the naming problems.
---

# Naming Check

Run scripts/check_naming.py (the tool is named check_naming) and write its output into the feedback.
Don't judge by yourself, and don't add or drop any item. Copy every line it prints.

Shared rules: check assigned variables, function names and function parameters; for repeated names, list only the first definition; don't check class names; skip self and cls.
A name may have one leading underscore, followed by a lowercase English letter, and then lowercase English letters, digits or underscores.
This exercise has no exception for constants: MAX fails; result_2 passes. The script runs these rules.

Output format: copy every line the script prints, for example "Line 3: myVar breaks the rule (suggest my_var)"
''',
    'skills/naming-check/SKILL.version-a.md': r'''---
name: naming-check
description: Checks whether the variable and function names in a Python homework are lowercase English with underscores (snake_case). Use it when reviewing a homework and you need to list the naming problems.
---

# Naming Check

Check assigned variables, function names and function parameters, and list every name that breaks the rule.
Name format: one leading underscore is allowed, followed by a lowercase English letter, and then lowercase English letters, digits or underscores.
For repeated names, list only the place of the first definition; don't check class names; skip self and cls.
This exercise has no exception for constants: MAX fails; result_2 passes.
If no name breaks the rule, answer "OK: all names follow this course's rules".

Output format: one line per problem, for example "Line 3: myVar"
''',
}
for rel, content in _files.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# Make Python look again for the files we just wrote (if another cell ran first, Python remembers "this folder does not exist")
import importlib
importlib.invalidate_caches()

# 3. Remember the model source (the API key is not saved) and test the connection
import llm
llm.save_settings(model_source, colab_model_name.strip(), gemini_model_name.strip())
if INSTRUCTOR_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = INSTRUCTOR_API_KEY.strip()
from workshop import *
model = prepare()

reply = model.invoke("Say hello in one sentence to a classmate who just learned Python").text
print(f"✅ Setup complete. Model source: {model_source}")
print(f"🤖 The model says: {reply.strip()}")

---
## Part 1: One job, three places it can live

> 📍 **Matching slides**: slide 14 "One naming check, three placements", slide 16 "Placement B: the model picks a tool, a script checks"

Before comparing, prepare two things:
1. This notebook uses the **same homework** from start to finish (printed below, with line numbers on the left).
2. The naming-check **script** `check_naming.py`: the first setup cell after you look at the homework writes it into the handbook's `scripts/` folder.
   It does just one thing and prints a few short lines, like a small command-line tool.
   The next setup cell is `tools/check_naming.py`: it wraps the script in `@tool`, turning it into a **tool** the AI can call,
   so it can go into the toolbox of an **agent** (an AI that decides for itself whether to use tools). Part 3 uses it.

### 🤔 Predictions
All three versions use the same rule:
- Check assigned variables, function names, and function parameters; for a repeated name, list only the place where it is first defined; do not check class names; skip `self` and `cls`.
- A name may have one leading underscore, then a lowercase letter, then lowercase letters, digits, or underscores.
- This exercise has no exception for constants, so `MAX` fails and `result_2` passes. This is this course's rule; it does not claim every Python project works this way.

The next cell only shows the homework. **Circle the names that break the rule and count them first, then check below.**

In [ ]:
#@title 👀 Look at the homework and answer on paper first (this cell does not show the answer)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
for i, line in enumerate(sample_homework["Inconsistent naming"].splitlines(), 1):
    print(f"{i:>3} | {line}")
print("Circle the names that break the rule first, then run the script below to check.")

In [ ]:
%%writefile /content/review-bot/skills/naming-check/scripts/check_naming.py
"""Check whether variable and function names are lowercase English with underscores (snake_case).

Usage:
    python check_naming.py homework.py        # check one file
    python check_naming.py < homework.py      # or read from standard input
Output:
    One line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)"
    When everything passes it prints "OK: all variable and function names are lowercase with underscores"

A script does one thing and prints only a few short lines, so people and the AI can both check it. The same input and the same script give the same result.
"""

import ast
import re
import sys

RULE = re.compile(r"^_?[a-z][a-z0-9_]*$")


def suggest_name(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def find_all_names(tree: ast.AST) -> dict[str, int]:
    """name → line number of its first appearance (variables, functions, parameters; class names are not included)."""
    names = {}

    def note(name, line):
        if name not in names or line < names[name]:
            names[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            note(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            note(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            note(node.arg, node.lineno)
    return names


def check_names(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"Cannot check: line {e.lineno} has a syntax error"
    problems = [
        f"Line {line}: {name} breaks the rule (suggest {suggest_name(name)})"
        for name, line in sorted(find_all_names(tree).items(), key=lambda x: (x[1], x[0]))
        if not RULE.match(name)
    ]
    return "\n".join(problems) if problems else "OK: all variable and function names are lowercase with underscores"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(check_names(source))


In [ ]:
%%writefile /content/review-bot/tools/check_naming.py
"""check_naming: wraps the Skill's script scripts/check_naming.py as an agent tool.

Wrap the script in @tool and it joins the agent's toolbox.
In this example the AI decides whether to call the tool and tidies up the result. With the same version and the same input, the Python script returns the same check result,
but you still need to confirm whether the model actually called it and whether it kept the result intact.
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

SCRIPT = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def run_naming_script(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(SCRIPT)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    output = proc.stdout.decode("utf-8").strip()
    return output or "The script failed to run: " + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """Check whether variable and function names are lowercase with underscores (snake_case), and return the lines that break the rule.

    When to use it: when reviewing a homework and you need to list the naming problems (this is the script that comes with the naming-check handbook).
    Parameter code: the complete Python code (a string).
    Returns: one line per problem, for example "Line 3: myVar breaks the rule (suggest my_var)". If everything passes it returns "OK: ...".
    """
    return run_naming_script(code)


In [ ]:
#@title ▶ Run: look at the homework, then run check_naming.py once from the command line
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import subprocess

homework = sample_homework["Inconsistent naming"]
(BASE / "homework.py").write_text(homework, encoding="utf-8")
print("This homework (line numbers on the left):")
for i, line in enumerate(homework.splitlines(), start=1):
    print(f"{i:>3} | {line}")

script = BASE / "skills" / "naming-check" / "scripts" / "check_naming.py"
print("\n$ python check_naming.py homework.py")
result = subprocess.run([sys.executable, str(script), str(BASE / "homework.py")],
                        capture_output=True, text=True, encoding="utf-8")
print(result.stdout.strip() or result.stderr.strip())
record("2-B Part 1: output of check_naming.py", result.stdout.strip())

### 👀 What did you see

- The script caught **6**: `CalcAverage`, `scoreList`, `Total`, `avgScore`, `myScores`, `MAX`. `s` and `result_2` pass.
- Does `MAX` count? By Python convention constants may be all caps, but our rule says "lowercase plus underscores", so **the script follows the rule**.
  Whether to allow an exception is a human decision; the same script on the same input gives the same result.
- The output is a few lines, one problem per line: the AI can read it and so can a person. This is the script principle: one script does one thing, and its output is short.

---
## Part 2: Placement A: written into the handbook text

> 📍 **Matching slides**: slide 15 "Placement A: the model judges from the handbook text"

The script just listed 6 problems under this course's rule. This part tries the most intuitive placement: write the rule into the handbook text and let the AI check it itself.

The next cell writes the handbook `skills/naming-check/SKILL.version-a.md` as the "text version": the same scope of names, format, and exception rules as the script.
The cell after it has the AI read this handbook and check the same homework, **three times in a row**.

### 🤔 Predictions
Will the three runs catch the same number? Will they match the 6 the script caught in Part 1?

In [ ]:
%%writefile /content/review-bot/skills/naming-check/SKILL.version-a.md
---
name: naming-check
description: Checks whether the variable and function names in a Python homework are lowercase English with underscores (snake_case). Use it when reviewing a homework and you need to list the naming problems.
---

# Naming Check

Check assigned variables, function names and function parameters, and list every name that breaks the rule.
Name format: one leading underscore is allowed, followed by a lowercase English letter, and then lowercase English letters, digits or underscores.
For repeated names, list only the place of the first definition; don't check class names; skip self and cls.
This exercise has no exception for constants: MAX fails; result_2 passes.
If no name breaks the rule, answer "OK: all names follow this course's rules".

Output format: one line per problem, for example "Line 3: myVar"


In [ ]:
#@title ▶ Run: Placement A, same homework three times (the AI checks by itself)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json, re

def read_handbook_body(path):
    text = path.read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text

def extract_names(text):
    """Pull "Line N: name" out of a reply and return a sorted [(line_no, name), ...]."""
    return sorted({(int(line_no), name) for line_no, name in re.findall(r"Line\s*(\d+)\s*:\s*([A-Za-z_]\w*)", text)})

def parse_status(text):
    lines = [line for line in text.splitlines() if line.strip()]
    unmatched = [line for line in lines if re.search(r"\bLine\s*\d", line) and not extract_names(line)]
    if unmatched:
        return "Some lines could not be parsed; check the original text"
    if not extract_names(text) and not text.strip().startswith("OK:"):
        return "Cannot confirm the number of problems from the format; check the original text"
    return "Parsable"

def save_result(version, data):
    path = BASE / "2B_results.json"
    all_results = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    all_results[version] = data
    path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding="utf-8")
from langchain_core.messages import HumanMessage, SystemMessage

handbook = read_handbook_body(BASE / "skills" / "naming-check" / "SKILL.version-a.md")
print("Handbook body the AI read:\n" + handbook + "\n")
homework = sample_homework["Inconsistent naming"]

counter = ModelCallCounter()
three_runs, evidence = [], []
for i in range(1, 4):
    answer = model.invoke(
        [SystemMessage(handbook), HumanMessage(f"Please check this homework:\n```python\n{homework}```")],
        config={"callbacks": [counter]},
    ).text
    print(f"Run {i} raw answer:\n{answer}")
    print("Parse status:", parse_status(answer))
    evidence.append({"raw_answer": answer, "parse_status": parse_status(answer)})
    result = extract_names(answer)
    three_runs.append(result)
    print(f"Run {i}: caught {len(result)} -> " + (", ".join(f"Line {n}: {name}" for n, name in result) or "(no parsable items; read the raw answer)"))

same = all(x == three_runs[0] for x in three_runs)
print(f"\nAre the three runs the same? {'✅ Same' if same else '❌ Different'}; the AI was called {counter.count} times in total")
save_result("A", {"three_runs": three_runs, "ai_calls": counter.count, "evidence": evidence})
record("2-B Part 2: Placement A (written in the handbook text)", {"three_runs": three_runs, "same": same, "ai_calls": counter.count, "evidence": evidence})

### 👀 What did you see

- Check whether the three results are the same, then compare them with the script's result. You may see missed items, extra items, different line numbers, or perfect agreement.
  The model judges from the text rule afresh each time, so the three runs may agree or may not.
- Three identical runs is also a valid observation; record it honestly and do not rerun until it goes wrong. Three identical runs is no guarantee for every input; also check that the names and line numbers fit the shared rule.
- And it may not match the script's 6. First separate the cases: the model judged wrongly, the reply format could not be parsed, or this run was judged correctly.

---
## Part 3: Placement B: a small program, and the handbook tells the AI to run it

> 📍 **Matching slides**: slide 16 "Placement B: the model picks a tool, a script checks"

In Placement A the AI re-judges from the text rule every time. This part gives the check to a script, and the AI only decides whether to run it.

The next cell writes a separate "script version" handbook `SKILL.md` (A uses a different file, so rerunning does not mix them): "Run `scripts/check_naming.py` (tool name `check_naming`) and write its output into the feedback."
The cell after it uses `create_agent` (the one-line way to make an agent) to build an agent: the handbook is its **system prompt**
(the text always placed first that tells the AI what to obey), and its toolbox holds just one tool, `check_naming` (the script wrapped in Part 1), again **run three times**.

### 🤔 Predictions
1. Will the three runs catch the same this time? Will they match the script's output from Part 1?
2. What two things does the AI do this time? Does it still need to think up "how to check"?

In [ ]:
%%writefile /content/review-bot/skills/naming-check/SKILL.md
---
name: naming-check
description: Checks whether the variable and function names in a Python homework are lowercase English with underscores (snake_case). Use it when reviewing a homework and you need to list the naming problems.
---

# Naming Check

Run scripts/check_naming.py (the tool is named check_naming) and write its output into the feedback.
Don't judge by yourself, and don't add or drop any item. Copy every line it prints.

Shared rules: check assigned variables, function names and function parameters; for repeated names, list only the first definition; don't check class names; skip self and cls.
A name may have one leading underscore, followed by a lowercase English letter, and then lowercase English letters, digits or underscores.
This exercise has no exception for constants: MAX fails; result_2 passes. The script runs these rules.

Output format: copy every line the script prints, for example "Line 3: myVar breaks the rule (suggest my_var)"


In [ ]:
#@title ▶ Run: Placement B, same homework three times (the AI decides whether to run the script)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json, re

def read_handbook_body(path):
    text = path.read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text

def extract_names(text):
    """Pull "Line N: name" out of a reply and return a sorted [(line_no, name), ...]."""
    return sorted({(int(line_no), name) for line_no, name in re.findall(r"Line\s*(\d+)\s*:\s*([A-Za-z_]\w*)", text)})

def parse_status(text):
    lines = [line for line in text.splitlines() if line.strip()]
    unmatched = [line for line in lines if re.search(r"\bLine\s*\d", line) and not extract_names(line)]
    if unmatched:
        return "Some lines could not be parsed; check the original text"
    if not extract_names(text) and not text.strip().startswith("OK:"):
        return "Cannot confirm the number of problems from the format; check the original text"
    return "Parsable"

def save_result(version, data):
    path = BASE / "2B_results.json"
    all_results = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    all_results[version] = data
    path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding="utf-8")
from langchain.agents import create_agent
from tools.check_naming import check_naming

handbook = read_handbook_body(BASE / "skills" / "naming-check" / "SKILL.md")
print("Handbook body the AI read:\n" + handbook + "\n")
homework = sample_homework["Inconsistent naming"]
helper = create_agent(model, tools=[check_naming], system_prompt=handbook)

counter = ModelCallCounter()
three_runs, evidence = [], []
for i in range(1, 4):
    result = helper.invoke(
        {"messages": [{"role": "user", "content": f"Please check this homework:\n```python\n{homework}```"}]},
        config={"callbacks": [counter]},
    )
    msgs = result["messages"]
    used = [c["name"] for m in msgs for c in (getattr(m, "tool_calls", None) or [])]
    tool_results = [m for m in msgs if m.type == "tool" and getattr(m, "name", "") == "check_naming"]
    script_output = [m.text for m in tool_results]
    final_reply = msgs[-1].text
    names_found = extract_names(final_reply)
    script_items = sorted({pair for t in script_output for pair in extract_names(t)})
    ran = any(getattr(m, "status", "success") != "error" for m in tool_results)
    kept = ran and names_found == script_items and parse_status(final_reply) == "Parsable"
    evidence.append({"tools_requested": used, "tool_returned_ok": ran, "raw_script_output": script_output, "script_items": script_items,
                     "final_reply": final_reply, "parse_status": parse_status(final_reply), "kept_all_items": kept})
    three_runs.append(names_found)
    print(f"========== Run {i} ==========")
    print("Tools the AI requested:", ", ".join(used) or "(no tool used; it judged by itself)")
    print("Tool returned successfully:", ran)
    if script_output:
        print("Script output this run:\n" + "\n---\n".join(script_output))
    print("The AI's final reply, verbatim:\n" + final_reply)
    print("Parse status:", parse_status(final_reply), "| Kept every script item:", kept)
    print(f"The AI's final reply yielded {len(names_found)} -> " + (", ".join(f"Line {n}: {name}" for n, name in names_found) or "(no parsable items; read the raw answer)"))
    print()

same = all(x == three_runs[0] for x in three_runs)
print(f"Are the three runs the same? {'✅ Same' if same else '❌ Different'}; the AI was called {counter.count} times in total")
save_result("B", {"three_runs": three_runs, "ai_calls": counter.count, "evidence": evidence})
record("2-B Part 3: Placement B (written as a script)", {"three_runs": three_runs, "same": same, "ai_calls": counter.count, "evidence": evidence})

### 👀 What did you see

- First confirm that the tool was really called in all three runs; when it was, compare the script's result with the final reply. A fixed script result does not mean the model calls it or relays it in full every time.
- The AI no longer works out how to check. Here we want it to do two things: **decide whether to run the tool** and **put the result into the reply**. Use the output to confirm what it actually did. With the same input and rules the script result is fixed, but the relaying still needs checking.
- But "whether to run it" is still the AI's decision: if in some run it did not call the tool and judged by itself, then even if the answer happens to match there is no evidence the script ran. Placement C in the next part does not have this problem.
- If the tool is called just once, there are usually two model responses: first it requests the tool, then it writes the final reply. With no tool use or several calls the count differs; go by the counter.

---
## Part 4: Placement C: a LangGraph node, and the AI never touches it

> 📍 **Matching slides**: slide 17 "Placement C: the workflow always runs the same check"

In Placement B the AI still decides whether to run the script. This part keeps the AI out of it altogether.

This version has no handbook and no agent, only a **flow** drawn with LangGraph. Each station in the flow is a **node** (just a function) and always runs when the flow reaches it;
the dict passed between stations is the **State**. This flow has only one station, `check_naming`: it calls the same script directly and writes the result into the State's `naming_problems`.
Like the helper's `check_syntax`: when the flow reaches this station, the check always happens.

### 🤔 Predictions
Will the three runs be the same? How many AI calls will this version make in total?

In [ ]:
#@title ▶ Run: Placement C, the flow always runs the check, three times
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json, re

def read_handbook_body(path):
    text = path.read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text

def extract_names(text):
    """Pull "Line N: name" out of a reply and return a sorted [(line_no, name), ...]."""
    return sorted({(int(line_no), name) for line_no, name in re.findall(r"Line\s*(\d+)\s*:\s*([A-Za-z_]\w*)", text)})

def parse_status(text):
    lines = [line for line in text.splitlines() if line.strip()]
    unmatched = [line for line in lines if re.search(r"\bLine\s*\d", line) and not extract_names(line)]
    if unmatched:
        return "Some lines could not be parsed; check the original text"
    if not extract_names(text) and not text.strip().startswith("OK:"):
        return "Cannot confirm the number of problems from the format; check the original text"
    return "Parsable"

def save_result(version, data):
    path = BASE / "2B_results.json"
    all_results = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    all_results[version] = data
    path.write_text(json.dumps(all_results, ensure_ascii=False, indent=2), encoding="utf-8")
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from tools.check_naming import run_naming_script

class NamingState(TypedDict):
    code: str
    naming_problems: str

def check_naming(state: NamingState) -> dict:
    return {"naming_problems": run_naming_script(state["code"])}

builder = StateGraph(NamingState)
builder.add_node("check_naming", check_naming)
builder.add_edge(START, "check_naming")
builder.add_edge("check_naming", END)
flow = builder.compile()

homework = sample_homework["Inconsistent naming"]
counter = ModelCallCounter()
three_runs = []
for i in range(1, 4):
    state = flow.invoke({"code": homework, "naming_problems": ""}, config={"callbacks": [counter]})
    result = extract_names(state["naming_problems"])
    three_runs.append(result)
    print(f"Run {i}: caught {len(result)} -> " + ", ".join(f"Line {n}: {name}" for n, name in result))

same = all(x == three_runs[0] for x in three_runs)
print(f"\nAre the three runs the same? {'✅ Same' if same else '❌ Different'}; the AI was called {counter.count} times in total")
save_result("C", {"three_runs": three_runs, "ai_calls": counter.count})
record("2-B Part 4: Placement C (written as a flow node)", {"three_runs": three_runs, "same": same, "ai_calls": counter.count})

### 👀 What did you see

- With the same input and script, these three runs should give the same result; the naming check itself makes **0 AI calls**. The model connection test in the environment setup is counted separately.
- B and C use the same script; keep the difference clear: **in B the AI decides whether to run it; in C the flow always calls it.**
  Whether this job may ever be skipped decides whether you choose B or C.

---
## Part 5: The three versions side by side

> 📍 **Matching slides**: slide 17 (the difference between B and C), slide 18 "Two questions, decided separately"

Below, we call Placements A, B, and C "the three versions". All three have run, and their results are saved to a file. This cell puts the results of Parts 2-4 side by side; a version you have not run yet will tell you to go back and run it.

### 🤔 Predictions
If all three versions give the same result, where is the difference? Which uses the least AI? If every homework must have its naming checked, would you pick B or C?

In [ ]:
#@title ▶ Run: compare the three versions side by side
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import json
from IPython.display import Markdown, display

path = BASE / "2B_results.json"
all_results = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
descriptions = {
    "A": ("A: written in the handbook text", "Part 2"),
    "B": ("B: written as a script, the AI decides to run it", "Part 3"),
    "C": ("C: written as a flow node, always runs", "Part 4"),
}
rows = ["| Version | Parsable items in the three runs | Same in all three | AI calls | Parse / tool evidence |", "| --- | --- | --- | --- | --- |"]
for version, (name, part) in descriptions.items():
    if version not in all_results:
        rows.append(f"| {name} | (not run yet: run {part} first) | | | |")
        continue
    three_runs = all_results[version]["three_runs"]
    counts = ", ".join(str(len(x)) for x in three_runs)
    same = "✅ Same" if all(x == three_runs[0] for x in three_runs) else "❌ Different"
    evidence = all_results[version].get("evidence", [])
    summary = "; ".join(e["parse_status"] + (f" / tool returned={e['tool_returned_ok']}, kept all={e['kept_all_items']}" if version == "B" else "") for e in evidence) or "The flow got the script result directly"
    rows.append(f"| {name} | {counts} | {same} | {all_results[version]['ai_calls']} | {summary} |")
display(Markdown("\n".join(rows)))
record("2-B Part 5: the three versions side by side", all_results)

### 👀 What did you see

- **A** is judged by the model, so record its consistency from the actual results; for **B**, look separately at whether the tool ran, the script output, and the model's relaying; **C** runs the script directly in the flow.
- **C**'s check makes no model call and is always run by the flow; **B** lets the model choose the tool as it sees fit. Three identical runs is an observation; how it runs is decided by the program's structure.

### 📌 Ask two questions separately

> **How is it done?** Fixed rules go in code or a script; criteria that need judgment go in the handbook.
> **Who decides to run it?** A must-do on every run is called by the flow every time; when it depends on the situation, leave it to the agent to pick a tool.

The same script can be called by the flow, and it can also be wrapped as a tool. These two questions are not three mutually exclusive drawers.

---
## Part 6: Pause: where does each of five jobs go?

> 📍 **Matching slides**: slide 19 "Pause: five tasks, two questions each"

Finally, answer in order: **What does the job use? Who decides whether it runs?** When the conditions are not enough, write down your assumption; answers can combine, and you need not pick just one.

| The job | How it is done / who decides to run it? |
| --- | --- |
| 1. What tone the feedback uses | ____ |
| 2. Run the homework and read the output | ____ |
| 3. At most three generations (including the first) | ____ |
| 4. Check whether comments were written | ____ |
| 5. Judge whether the logic is well written | ____ |

<details>
<summary>Open the reference answers after you write yours</summary>

| The job | How it is done | Who runs it, and why |
| --- | --- | --- |
| 1. What tone the feedback uses | **Handbook** | Write the tone guidance into a Skill and give it to the model when it writes feedback |
| 2. Run the homework and read the output | **Script / tool** | Run it with a tool; this example lets the agent use it as it sees fit, and if every submission must be tested, have the flow arrange it instead |
| 3. At most three generations (including the first) | **Code** | In this course `retries` counts the first draft, and `retries >= 3` stops; it can be written as an if and must not be missed, or it would loop forever |
| 4. Check whether comments were written | **Script** (make it a **flow node** if it must never be missed) | Use code to recognize real comments; if every submission must be checked, the flow arranges it, and an optional check can be wrapped as a tool. A # inside a string is not a comment |
| 5. Judge whether the logic is well written | **Handbook** | Put the quality criteria into a Skill; correctness tests that can run still go to code, and the model adds interpretation |

</details>

## Wrap-up: save your work and hand it in

- 💾 **Save to Google Drive**: save a copy of `/content/review-bot/` (your edited handbooks, code, and State records) to your own Google Drive.
- ⬇️ **Download the zip**: download it straight to your computer.
- 📸 **The last cell**: prints the results the experiment cells saved. Pick the relevant records for a screenshot, then add your observations and reasons as the assignment asks.

These are the results the experiment cells saved through `record()`, not every intermediate state of every run.
Records in the same runtime pile up, so they may include experiments from other notebooks; use the printed titles to tell them apart.

A temporary browser disconnect does not necessarily wipe your files; files under `/content/` are lost only when the runtime is deleted or recycled.
To keep your handbooks, code, and checkpoint, back them up first. The notebook itself is saved with the menu "File → Save a copy in Drive". The two kinds of saving do different jobs.
When you come back, first check that the runtime is still there. If it was reset, re-run "0. Setup" and this part's setup cell, then restore the files you backed up as needed.

In [ ]:
#@title 💾 Save to Google Drive (an authorization window will pop up)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
save_to_drive()

In [ ]:
#@title ⬇️ Download the zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
download_zip()

In [ ]:
#@title 📸 Print the recorded experiment results (pick the relevant records to hand in)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
print_history()